# Replicating Table 4 — Full Evaluation of 9 Forecasters × 24 Assets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Conformal_Oracle/blob/main/python/examples/notebooks/reproduce_table4_full.ipynb)

This notebook replicates **Table 4** of Pele et al. (2026): the master evaluation
table comparing 9 forecasters across 24 assets at the 1% VaR level, before and
after conformal recalibration.

**Panel A (Genuine recalibration):** Moirai 1.1, Lag-Llama, GJR-GARCH, GARCH-N, Hist. Sim.  
**Panel B (Effective replacement):** Chronos-Small, Chronos-Mini, TimesFM 2.5, Moirai 2.0

**Runtime estimates (Colab T4 GPU):**
- Base forecasters (GJR-GARCH, GARCH-N, Hist. Sim.): ~45 min
- TSFM forecasters (5 models): ~3–4 hours
- Total: ~4–5 hours

Results are checkpointed after each forecaster so you can resume or inspect
partial results.

> **Note:** The paper uses EWMA as one of the 10 models, but the `conformal-oracle`
> package does not yet include an EWMA forecaster. This notebook reproduces the
> remaining 9 models.

In [ ]:
!pip install -q conformal-oracle[all]
!pip install -q git+https://github.com/time-series-foundation-models/lag-llama.git
!pip install -q git+https://github.com/google-research/timesfm.git

import sys
if "google.colab" in sys.modules:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import numpy as np
import pandas as pd
import time
import json
from pathlib import Path
from conformal_oracle import audit
from conformal_oracle.contrib.benchmarks import (
    GJRGARCHForecaster,
    GARCHNormalForecaster,
    HistoricalSimulationForecaster,
)

print(f"conformal-oracle ready")

In [ ]:
# Clone repo for bundled return data (24 assets, 2000-2026)
import os, subprocess
if not Path("Conformal_Oracle").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/danpele/Conformal_Oracle.git"], check=True)

DATA_DIR = Path("Conformal_Oracle/cfp_ijf_data/returns")
ASSETS = sorted([f.stem for f in DATA_DIR.glob("*.csv") if "2" not in f.stem])
print(f"{len(ASSETS)} assets: {ASSETS}")

returns = {}
for asset in ASSETS:
    df = pd.read_csv(DATA_DIR / f"{asset}.csv", index_col="date", parse_dates=True)
    returns[asset] = df["log_return"]
    returns[asset].name = asset

print(f"\nSample lengths: min={min(len(v) for v in returns.values())}, "
      f"max={max(len(v) for v in returns.values())}")

In [ ]:
ALPHA = 0.01
CHECKPOINT = Path("table4_results.json")

def load_checkpoint():
    if CHECKPOINT.exists():
        with open(CHECKPOINT) as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT, "w") as f:
        json.dump(results, f, indent=2)

def run_forecaster(name, forecaster, warmup=50):
    """Run audit on all 24 assets for one forecaster."""
    all_results = load_checkpoint()
    if name in all_results:
        print(f"{name}: already computed ({len(all_results[name])} assets), skipping.")
        return

    asset_results = []
    t0 = time.time()
    for i, asset in enumerate(ASSETS):
        t_asset = time.time()
        try:
            result = audit(returns[asset], forecaster, alpha=ALPHA, mode="static", warmup=warmup)
            d = result.to_dict()
            var_width = float(np.mean(result.var_corrected))
            asset_results.append({
                "asset": asset,
                "pihat_raw": d["violation_rate_raw"],
                "pihat_cp": d["violation_rate_corrected"],
                "p_kup_raw": d["kupiec_pvalue_raw"],
                "p_kup_cp": d["kupiec_pvalue_corrected"],
                "QS_raw": d["quantile_score_raw"],
                "QS_cp": d["quantile_score_corrected"],
                "VaR_width": var_width,
                "TL_cp": d["basel_zone_corrected"],
            })
            elapsed = time.time() - t_asset
            print(f"  [{i+1:2d}/24] {asset:12s} done  ({elapsed:.0f}s)  "
                  f"pi_raw={d['violation_rate_raw']:.3f}  pi_cp={d['violation_rate_corrected']:.3f}")
        except Exception as e:
            print(f"  [{i+1:2d}/24] {asset:12s} FAILED: {e}")
            asset_results.append({"asset": asset, "error": str(e)})

    all_results[name] = asset_results
    save_checkpoint(all_results)
    total = time.time() - t0
    print(f"{name}: completed 24 assets in {total/60:.1f} min\n")

print("Checkpoint system ready.")

## Panel A — Base Forecasters

These require no extra dependencies.

In [ ]:
print("=== GJR-GARCH ===")
run_forecaster("GJR-GARCH", GJRGARCHForecaster(window=250))

print("=== GARCH-N ===")
run_forecaster("GARCH-N", GARCHNormalForecaster(window=250))

print("=== Hist. Sim. ===")
run_forecaster("Hist-Sim", HistoricalSimulationForecaster())

## Panel A — TSFM Forecasters (Genuine Recalibration)

Require GPU runtime for reasonable speed. Each cell runs independently
and checkpoints results.

In [ ]:
from conformal_oracle.contrib.tsfm.moirai import MoiraiForecaster

print("=== Moirai 1.1 ===")
run_forecaster("Moirai-1.1", MoiraiForecaster(version="1.1"), warmup=512)

print("=== Moirai 2.0 ===")
run_forecaster("Moirai-2.0", MoiraiForecaster(version="2.0"), warmup=512)

In [ ]:
from conformal_oracle.contrib.tsfm.lag_llama import LagLlamaForecaster

print("=== Lag-Llama ===")
run_forecaster("Lag-Llama", LagLlamaForecaster(), warmup=512)

In [ ]:
from conformal_oracle.contrib.tsfm.chronos import ChronosForecaster

print("=== Chronos-Small ===")
run_forecaster("Chronos-Small", ChronosForecaster(size="small"), warmup=512)

print("=== Chronos-Mini ===")
run_forecaster("Chronos-Mini", ChronosForecaster(size="mini"), warmup=512)

In [ ]:
from conformal_oracle.contrib.tsfm.timesfm import TimesFM25Forecaster

print("=== TimesFM 2.5 ===")
run_forecaster("TimesFM-2.5", TimesFM25Forecaster(), warmup=512)

## Aggregation — Table 4

In [ ]:
all_results = load_checkpoint()
print(f"Models computed: {list(all_results.keys())}\n")

MODEL_ORDER = ["Moirai-1.1", "Lag-Llama", "GJR-GARCH", "GARCH-N", "Hist-Sim",
               "Chronos-Small", "Chronos-Mini", "TimesFM-2.5", "Moirai-2.0"]

rows = []
for model in MODEL_ORDER:
    if model not in all_results:
        continue
    assets = [r for r in all_results[model] if "error" not in r]
    n = len(assets)
    if n == 0:
        continue
    df_m = pd.DataFrame(assets)

    gjr_width = None
    if "GJR-GARCH" in all_results:
        gjr_assets = [r for r in all_results["GJR-GARCH"] if "error" not in r]
        gjr_width = pd.DataFrame(gjr_assets)["VaR_width"].mean()

    width_mean = df_m["VaR_width"].mean()
    rows.append({
        "Model": model,
        "π̂ Raw": df_m["pihat_raw"].mean(),
        "π̂ Corr.": df_m["pihat_cp"].mean(),
        "Kupiec Raw": f"{int((df_m['p_kup_raw'] >= 0.05).sum())}/{n}",
        "Kupiec Corr.": f"{int((df_m['p_kup_cp'] >= 0.05).sum())}/{n}",
        "QS Raw (×10⁴)": df_m["QS_raw"].mean() * 1e4,
        "QS Corr. (×10⁴)": df_m["QS_cp"].mean() * 1e4,
        "Width": width_mean,
        "W/GJR": width_mean / gjr_width if gjr_width else np.nan,
        "Green": f"{int((df_m['TL_cp'] == 'green').sum())}/{n}",
    })

table4 = pd.DataFrame(rows).set_index("Model")
display(table4.style.format({
    "π̂ Raw": "{:.3f}", "π̂ Corr.": "{:.3f}",
    "QS Raw (×10⁴)": "{:.1f}", "QS Corr. (×10⁴)": "{:.1f}",
    "Width": "{:.3f}", "W/GJR": "{:.2f}",
}))

## Comparison with Paper Table 4

Reference values from Pele et al. (2026), Table 4:

| Model | π̂ Raw | π̂ Corr. | Kupiec Raw | Kupiec Corr. | QS Raw | QS Corr. | Width | W/GJR | Green |
|:------|-------:|--------:|-----------:|-------------:|-------:|---------:|------:|------:|------:|
| **Panel A: Genuine recalibration** | | | | | | | | | |
| Moirai 1.1 | .015 | .011 | 13/24 | 15/24 | 5.4 | 5.3 | .039 | 1.00 | 21/24 |
| Lag-Llama | .029 | .009 | 0/24 | 22/24 | 5.9 | 5.2 | .041 | 1.05 | 24/24 |
| GJR-GARCH | .004 | .011 | 7/24 | 16/24 | 5.0 | 4.9 | .039 | 1.00 | 19/24 |
| GARCH-N | .019 | .010 | 8/24 | 18/24 | 4.9 | 4.7 | .036 | 0.93 | 21/24 |
| Hist. Sim. | .016 | .011 | 11/24 | 21/24 | 5.7 | 5.6 | .039 | 1.00 | 22/24 |
| **Panel B: Effective replacement** | | | | | | | | | |
| Chronos-Small | .388 | .011 | 0/24 | 14/24 | 37.9 | 5.8 | .040 | 1.03 | 19/24 |
| Chronos-Mini | .419 | .010 | 0/24 | 11/24 | 41.1 | 5.7 | .041 | 1.04 | 19/24 |
| TimesFM 2.5 | .990 | .013 | 0/24 | 3/24 | 334.0 | 9.5 | .069 | 1.75 | 19/24 |
| Moirai 2.0 | .988 | .015 | 0/24 | 4/24 | 304.9 | 9.1 | .063 | 1.60 | 19/24 |

Numerical differences reflect Monte Carlo variation in TSFM sampling and
minor differences in GARCH optimizer convergence across platforms.